# Sesión 03 — Sistemas OFDM e Implementación
### Laboratorio Python · Sistemas de Comunicaciones Inalámbricas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ollerenac/wireless-communication-systems/blob/main/docs/sessions/03-ofdm-systems/lab.ipynb)

## Objetivos del Laboratorio

1. Visualizar la estructura temporal y espectral de un símbolo OFDM
2. Verificar que la cadena IFFT → canal → FFT recupera los símbolos perfectamente cuando el canal es plano
3. Demostrar experimentalmente que sin CP aparece ISI e ICI, y que el CP las elimina
4. Implementar los ecualizadores ZF y MMSE y comparar su BER sobre canal frequency-selective
5. Medir la BER de OFDM sobre canal multipath y compararla con la de AWGN puro

In [ ]:
%pip install numpy scipy matplotlib --quiet

import os
import numpy as np
from scipy.special import erfc
import matplotlib.pyplot as plt

os.makedirs('figures', exist_ok=True)

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

rng = np.random.default_rng(seed=0)
print('Setup completado.')

## Repaso Teórico

### Cadena OFDM completa

```
TX: Bits → QAM mapper → [X_0, X_1, ..., X_{N-1}] → IFFT → añadir CP → canal h[l] → AWGN
RX: → eliminar CP → FFT → [Y_0, Y_1, ..., Y_{N-1}] → ecualizador H[k] → QAM demod → Bits
```

**Señal OFDM en tiempo:**
$$x[n] = \frac{1}{\sqrt{N}}\sum_{k=0}^{N-1} X[k]\, e^{j2\pi kn/N}, \quad n = 0, \ldots, N-1$$

**Tras eliminar CP y aplicar FFT con canal circular:**
$$Y[k] = H[k]\cdot X[k] + W[k]$$

donde $H[k] = \sum_l h[l]\, e^{-j2\pi kl/N}$ es la DFT del canal.

**Ecualizador ZF:**
$$\hat{X}^{ZF}[k] = \frac{Y[k]}{H[k]}$$

**Ecualizador MMSE:**
$$\hat{X}^{MMSE}[k] = \frac{H^*[k]}{|H[k]|^2 + 1/\text{SNR}}\cdot Y[k]$$

### Parámetros del sistema para este laboratorio

| Parámetro | Valor |
|-----------|-------|
| $N$ (subportadoras) | 64 |
| $N_{CP}$ (CP) | 16 muestras |
| Modulación | QPSK ($M=4$) |
| Canal | multipath (configurable) |

---
## Parámetros Globales del Sistema

In [ ]:
# Parámetros OFDM
N    = 64    # número de subportadoras
N_CP = 16   # longitud del prefijo cíclico
M    = 4    # orden de modulación (QPSK)
k    = int(np.log2(M))  # bits por símbolo

# Canal multipath de referencia: 3 caminos
h_channel = np.array([0.8, 0.0, 0.0, 0.5, 0.0, 0.0, 0.3])  # longitud L=7
h_channel = h_channel / np.linalg.norm(h_channel)            # normalizar energía
L = len(h_channel)

print(f'Parámetros OFDM: N={N}, N_CP={N_CP}, M={M} ({k} bits/símbolo)')
print(f'Canal: {L} caminos, energía normalizada')
print(f'Overhead CP: {N_CP/(N+N_CP)*100:.1f}%')
print(f'Condición CP: N_CP={N_CP} >= L-1={L-1}: {N_CP >= L-1}')

---
## Ejercicio 1 — Señal OFDM en Tiempo y Frecuencia
### ⏱ Tiempo estimado: 15 min

Genera un símbolo OFDM y visualiza su estructura en los dominios temporal y frecuencial.

In [ ]:
def qpsk_map(bits):
    """Mapeado QPSK Gray: 2 bits → 1 símbolo complejo (|s|² = 1)."""
    # 00→(+1+j), 01→(-1+j), 11→(-1-j), 10→(+1-j)
    b0, b1 = bits[0::2], bits[1::2]
    I = 1 - 2*b0          # 0→+1, 1→-1
    Q = 1 - 2*b1
    return (I + 1j*Q) / np.sqrt(2)  # energía media = 1


def ofdm_tx(X, N_CP):
    """Genera símbolo OFDM: IFFT + añadir CP."""
    x = np.fft.ifft(X, norm='ortho')       # IFFT normalizada: E[|x|²] = E[|X|²]
    cp = x[-N_CP:]                          # últimas N_CP muestras
    return np.concatenate([cp, x])          # [CP | símbolo]


# Genera un símbolo OFDM con N subportadoras QPSK aleatorias
bits_tx = rng.integers(0, 2, N * k)
X_tx = qpsk_map(bits_tx)                   # N símbolos QPSK
x_ofdm = ofdm_tx(X_tx, N_CP)              # símbolo OFDM con CP

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Panel izquierdo: dominio temporal
t = np.arange(len(x_ofdm))
axes[0].plot(t, x_ofdm.real, 'b-', lw=1.5, label='Parte real')
axes[0].axvspan(0, N_CP, alpha=0.2, color='orange', label=f'CP ({N_CP} muestras)')
axes[0].axvline(N_CP, color='orange', ls='--', lw=1)
axes[0].set_xlabel('Muestra $n$')
axes[0].set_ylabel('Amplitud')
axes[0].set_title('Símbolo OFDM en tiempo (parte real)')
axes[0].legend()

# Panel derecho: espectro de potencia
X_spectrum = np.fft.fft(x_ofdm[N_CP:], norm='ortho')  # FFT del símbolo sin CP
freqs = np.fft.fftfreq(N)
psd = np.abs(X_spectrum)**2
axes[1].stem(np.fft.fftshift(freqs), np.fft.fftshift(psd),
             linefmt='b-', markerfmt='bo', basefmt='gray')
axes[1].set_xlabel('Frecuencia normalizada ($f/f_s$)')
axes[1].set_ylabel('Potencia')
axes[1].set_title('Espectro de potencia por subportadora')

plt.tight_layout()
plt.savefig('figures/ofdm-time-domain.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print(f'Longitud total del símbolo transmitido: {len(x_ofdm)} muestras = N + N_CP = {N} + {N_CP}')
print(f'Potencia media de la señal OFDM: {np.mean(np.abs(x_ofdm)**2):.4f} (teórica: 1.0)')

### Preguntas de reflexión — Ejercicio 1

1. ¿Qué relación ves entre la región naranja (CP) y el final del símbolo OFDM? ¿Son idénticas?

2. El espectro de potencia muestra N picos. ¿Esperarías que todos tengan la misma potencia? ¿Por qué en la práctica varían?
   > *Pista: los símbolos QPSK tienen amplitud constante |s|=1, pero la DFT de una secuencia aleatoria no tiene potencia uniforme.*

3. Modifica el código para generar un símbolo con solo las 8 subportadoras centrales activas (el resto a cero). ¿Cómo cambia la señal en el dominio temporal?

---
## Ejercicio 2 — Cadena IFFT/FFT sin Canal
### ⏱ Tiempo estimado: 10 min

Verifica que la cadena TX → (sin canal) → RX recupera los símbolos exactamente.

In [ ]:
def ofdm_rx_no_channel(x_with_cp, N, N_CP):
    """Receptor OFDM: eliminar CP + FFT."""
    x = x_with_cp[N_CP:]                    # eliminar CP
    Y = np.fft.fft(x, norm='ortho')         # FFT
    return Y


# Sin canal: conectar TX directamente a RX
X_rx_ideal = ofdm_rx_no_channel(x_ofdm, N, N_CP)

# Comparar símbolos transmitidos y recibidos
error = np.max(np.abs(X_rx_ideal - X_tx))
print(f'Error máximo |X_rx - X_tx|: {error:.2e}')
print(f'Recuperación perfecta: {error < 1e-10}')

# Visualizar constelación
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(X_tx.real, X_tx.imag, s=40, alpha=0.7, label='Transmitido $X_k$')
axes[0].set_title('Constelación transmitida')
axes[0].set_xlabel('I'); axes[0].set_ylabel('Q')
axes[0].set_aspect('equal')

axes[1].scatter(X_rx_ideal.real, X_rx_ideal.imag, s=40, alpha=0.7, color='orange', label='Recibido $Y_k$ (sin canal)')
axes[1].set_title('Constelación recibida (sin canal)')
axes[1].set_xlabel('I'); axes[1].set_ylabel('Q')
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()
print('Las constelaciones son idénticas: la cadena IFFT → (sin canal) → FFT es invertible perfectamente.')

---
## Ejercicio 3 — ISI sin CP vs con CP
### ⏱ Tiempo estimado: 20 min

Observa el efecto del canal multipath sobre la constelación recibida, con y sin CP.

In [ ]:
def apply_channel(x_signal, h):
    """Convolución lineal: simula el canal multipath."""
    return np.convolve(x_signal, h, mode='full')[:len(x_signal)]


def ofdm_rx_with_channel(y_received, N, N_CP, h, equalize='zf', SNR_dB=30):
    """
    Receptor OFDM completo:
    1. Elimina CP
    2. FFT
    3. Ecualización ZF o MMSE usando H[k] perfecto (canal conocido)
    """
    x = y_received[N_CP:]               # eliminar CP
    Y = np.fft.fft(x, norm='ortho')    # FFT

    # DFT del canal (N puntos)
    H = np.fft.fft(h, n=N)             # H[k]

    if equalize == 'zf':
        X_hat = Y / H
    elif equalize == 'mmse':
        SNR_lin = 10 ** (SNR_dB / 10)
        X_hat = (np.conj(H) / (np.abs(H)**2 + 1/SNR_lin)) * Y
    else:
        X_hat = Y  # sin ecualización

    return X_hat


# Transmitir N_sym símbolos OFDM consecutivos
N_sym = 100
all_bits = rng.integers(0, 2, N_sym * N * k)

# --- CASO A: sin CP ---
X_hat_nocp_list = []
X_tx_list = []
prev_tail = np.zeros(L - 1, dtype=complex)  # cola del símbolo anterior

for s in range(N_sym):
    bits_s = all_bits[s * N * k:(s+1) * N * k]
    X_s = qpsk_map(bits_s)
    X_tx_list.append(X_s)
    # Sin CP: transmitir solo el símbolo OFDM
    x_s = np.fft.ifft(X_s, norm='ortho')
    # Canal: añadir ISI de la cola del símbolo anterior
    x_with_tail = np.concatenate([prev_tail, x_s])
    y_s = apply_channel(x_with_tail, h_channel)[L-1:]  # tomar las N muestras útiles
    prev_tail = x_s[-(L-1):]
    Y_s = np.fft.fft(y_s, norm='ortho')
    # Sin ecualización para ver el efecto del canal
    H_k = np.fft.fft(h_channel, n=N)
    X_hat_nocp_list.append(Y_s / H_k)

# --- CASO B: con CP ---
X_hat_cp_list = []
for s in range(N_sym):
    bits_s = all_bits[s * N * k:(s+1) * N * k]
    X_s = qpsk_map(bits_s)
    x_s_cp = ofdm_tx(X_s, N_CP)
    y_s_cp = apply_channel(x_s_cp, h_channel)
    y_s_cp = y_s_cp[:N + N_CP]  # longitud correcta
    X_hat_s = ofdm_rx_with_channel(y_s_cp, N, N_CP, h_channel, equalize='zf')
    X_hat_cp_list.append(X_hat_s)

X_tx_all   = np.concatenate(X_tx_list)
X_nocp_all = np.concatenate(X_hat_nocp_list)
X_cp_all   = np.concatenate(X_hat_cp_list)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, data, title in zip(axes,
    [X_tx_all[:500], X_nocp_all[:500], X_cp_all[:500]],
    ['Transmitido', 'Recibido (sin CP, ZF)', 'Recibido (con CP, ZF)']):
    ax.scatter(data.real, data.imag, s=3, alpha=0.4)
    ax.set_title(title)
    ax.set_xlabel('I'); ax.set_ylabel('Q')
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
    ax.set_aspect('equal')

plt.suptitle('Efecto del CP sobre la constelación recibida (canal multipath, sin ruido)', y=1.02)
plt.tight_layout()
plt.savefig('figures/cp-effect-constellation.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Calcular SER
def qpsk_decide(X):
    return np.sign(X.real).astype(int), np.sign(X.imag).astype(int)

I_tx, Q_tx = qpsk_decide(X_tx_all)
_, _ = qpsk_decide(X_nocp_all)
I_cp, Q_cp = qpsk_decide(X_cp_all)

ser_cp = np.mean((qpsk_decide(X_cp_all)[0] != I_tx) | (qpsk_decide(X_cp_all)[1] != Q_tx))
print(f'SER con CP (sin ruido): {ser_cp:.6f} (esperado: 0)')
print('Con CP y canal conocido, la recuperación es perfecta en ausencia de ruido.')

### Preguntas de reflexión — Ejercicio 3

1. En la constelación sin CP, ¿ves puntos aislados en las 4 esquinas o una nube dispersa? ¿Por qué la interferencia multipath produce este patrón?

2. Con CP, la constelación recuperada debería ser perfecta (SER=0 sin ruido). ¿Se cumple? ¿Qué ocurriría si el canal tuviera un camino con retardo mayor que N_CP?

3. Modifica el canal para que tenga un camino en el índice `l = N_CP + 2` (mayor que N_CP). ¿Qué le ocurre a la constelación recuperada?

---
## Ejercicio 4 — Ecualización ZF vs MMSE
### ⏱ Tiempo estimado: 20 min

Compara los ecualizadores ZF y MMSE en un canal con ganancia nula en algunas subportadoras.

In [ ]:
def simulate_ofdm_ber(EbN0_dB_range, N, N_CP, h, M=4, N_frames=500, equalize='zf'):
    """
    Simula BER de OFDM sobre canal multipath con ecualización ZF o MMSE.
    Canal perfecto conocido en el receptor.
    """
    k = int(np.log2(M))
    ber = np.zeros(len(EbN0_dB_range))

    for i, EbN0_dB in enumerate(EbN0_dB_range):
        SNR_lin = 10 ** (EbN0_dB / 10)
        # Potencia de ruido por muestra: sigma² = E_s/(k * SNR) = 1/(k*SNR)
        sigma2 = 1 / (k * SNR_lin)

        errors = 0
        total  = 0

        for _ in range(N_frames):
            # TX
            bits = rng.integers(0, 2, N * k)
            X = qpsk_map(bits)  # QPSK
            x_cp = ofdm_tx(X, N_CP)

            # Canal + ruido
            y_ch = apply_channel(x_cp, h)
            y_ch = y_ch[:N + N_CP]
            noise = (rng.normal(0, np.sqrt(sigma2/2), N + N_CP) +
                     1j * rng.normal(0, np.sqrt(sigma2/2), N + N_CP))
            y_noisy = y_ch + noise

            # RX
            X_hat = ofdm_rx_with_channel(y_noisy, N, N_CP, h,
                                         equalize=equalize, SNR_dB=EbN0_dB)

            # Detección QPSK
            bits_hat = np.zeros(N * k, dtype=int)
            bits_hat[0::2] = (X_hat.real < 0).astype(int)
            bits_hat[1::2] = (X_hat.imag < 0).astype(int)

            errors += np.sum(bits != bits_hat)
            total  += N * k

        ber[i] = errors / total

    return ber


def Q_func(x):
    return 0.5 * erfc(x / np.sqrt(2))


EbN0_dB = np.arange(0, 22, 2)
ber_zf   = simulate_ofdm_ber(EbN0_dB, N, N_CP, h_channel, equalize='zf')
ber_mmse = simulate_ofdm_ber(EbN0_dB, N, N_CP, h_channel, equalize='mmse')
ber_awgn = Q_func(np.sqrt(2 * 10**(EbN0_dB/10)))  # referencia AWGN + QPSK

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogy(EbN0_dB, ber_awgn, 'k--', lw=2, label='AWGN puro (QPSK teórica)')
ax.semilogy(EbN0_dB, ber_zf,   'r-o', lw=1.5, markersize=5, label='OFDM + ZF (canal multipath)')
ax.semilogy(EbN0_dB, ber_mmse, 'b-s', lw=1.5, markersize=5, label='OFDM + MMSE (canal multipath)')
ax.axhline(1e-3, color='gray', ls=':', lw=1)
ax.set_xlabel('$E_b/N_0$ (dB)')
ax.set_ylabel('BER')
ax.set_title('Ejercicio 4 — BER OFDM: ZF vs MMSE vs AWGN')
ax.legend()
ax.set_ylim(1e-4, 1)
plt.tight_layout()
plt.savefig('figures/ofdm-ber-equalizers.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Analizar ganancias del canal por subportadora
H_k = np.fft.fft(h_channel, n=N)
print(f'Ganancia mínima del canal: {np.min(np.abs(H_k)):.3f} (subportadora {np.argmin(np.abs(H_k))})')
print(f'Ganancia máxima del canal: {np.max(np.abs(H_k)):.3f} (subportadora {np.argmax(np.abs(H_k))})')

### Preguntas de reflexión — Ejercicio 4

1. ¿Qué ecualizador da menor BER a SNR alta? ¿Y a SNR baja? ¿Por qué?
   > *Pista: a SNR baja, el ZF amplifica el ruido en subportadoras con canal débil; el MMSE lo limita.*

2. La BER sobre canal multipath es mayor que sobre AWGN puro (para el mismo Eb/N0). ¿Por qué?
   > *Pista: las subportadoras con ganancia baja contribuyen desproporcionadamente a la BER total.*

3. Modifica el canal para que sea completamente plano: `h_channel = np.array([1.0])`. ¿Esperarías que la BER con ZF coincida con la de AWGN? ¿Por qué?

---
## Ejercicio 5 — BER de OFDM vs AWGN y Análisis del Canal
### ⏱ Tiempo estimado: 25 min

Analiza la respuesta en frecuencia del canal y su impacto en la BER por subportadora.

In [ ]:
# Respuesta en frecuencia del canal
H_k = np.fft.fft(h_channel, n=N)
H_mag_dB = 20 * np.log10(np.abs(H_k) + 1e-10)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Panel izquierdo: respuesta en frecuencia del canal
axes[0].stem(np.arange(N), H_mag_dB, linefmt='b-', markerfmt='bo', basefmt='gray')
axes[0].axhline(np.mean(H_mag_dB), color='r', ls='--', label=f'Media = {np.mean(H_mag_dB):.1f} dB')
axes[0].set_xlabel('Subportadora $k$')
axes[0].set_ylabel('$|H[k]|$ (dB)')
axes[0].set_title('Respuesta en frecuencia del canal')
axes[0].legend()

# Panel derecho: BER por subportadora a SNR = 15 dB
SNR_target = 15  # dB
snr_per_subcarrier_dB = H_mag_dB + SNR_target  # ganancia por subportadora
ber_per_sc = Q_func(np.sqrt(2 * 10**(snr_per_subcarrier_dB/10)))

axes[1].semilogy(np.arange(N), ber_per_sc, 'r-', lw=1.5)
axes[1].axhline(Q_func(np.sqrt(2 * 10**(SNR_target/10))), color='k', ls='--',
                label=f'BER AWGN a {SNR_target} dB')
axes[1].set_xlabel('Subportadora $k$')
axes[1].set_ylabel('BER (QPSK)')
axes[1].set_title(f'BER teórica por subportadora ($E_b/N_0 = {SNR_target}$ dB)')
axes[1].legend()

plt.tight_layout()
plt.savefig('figures/ofdm-per-subcarrier-ber.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# BER global vs AWGN a SNR = 15 dB
ber_global = np.mean(ber_per_sc)
ber_awgn_ref = Q_func(np.sqrt(2 * 10**(SNR_target/10)))
print(f'BER global OFDM sobre canal multipath a {SNR_target} dB: {ber_global:.2e}')
print(f'BER AWGN de referencia a {SNR_target} dB:                 {ber_awgn_ref:.2e}')
print(f'Degradación debida al canal: {10*np.log10(ber_global/ber_awgn_ref):.1f} dB equivalentes')

# Visualización de las figuras para apuntes
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Subportadoras OFDM en frecuencia
f = np.linspace(-3, 3, 2000)
colors = plt.cm.tab10(np.linspace(0, 1, 6))
for idx, sc in enumerate([-2, -1, 0, 1, 2, 3]):
    sinc_env = np.sinc(f - sc)
    axes[0].plot(f, np.abs(sinc_env), color=colors[idx], lw=1.5,
                 label=f'k={sc}' if abs(sc) <= 1 else None)
axes[0].set_xlabel('Frecuencia normalizada (múltiplos de $\\Delta f$)')
axes[0].set_ylabel('Amplitud espectral')
axes[0].set_title('Espectros de 6 subportadoras OFDM adyacentes')
axes[0].set_xlim(-3, 3)
# Marcar nulos
for sc in [-2, -1, 0, 1, 2, 3]:
    for other in [-2, -1, 0, 1, 2, 3]:
        if sc != other:
            axes[0].axvline(other, color='gray', ls=':', lw=0.5, alpha=0.5)

# CP illustration  
x_demo = np.fft.ifft(qpsk_map(rng.integers(0, 2, N*k)), norm='ortho').real
x_with_cp = np.concatenate([x_demo[-N_CP:], x_demo])
t_full = np.arange(len(x_with_cp))
axes[1].plot(t_full, x_with_cp, 'b-', lw=1.5)
axes[1].axvspan(0, N_CP, alpha=0.25, color='orange')
axes[1].axvspan(N + N_CP - N_CP, N + N_CP, alpha=0.1, color='orange')
axes[1].annotate('CP\n(copia)', xy=(N_CP/2, 0.6), ha='center', fontsize=9,
                 color='darkorange', fontweight='bold')
axes[1].annotate('', xy=(0.5, 0.5), xytext=(N + N_CP - N_CP + 0.5, 0.5),
                 arrowprops=dict(arrowstyle='<->', color='orange', lw=1.5))
axes[1].set_xlabel('Muestra $n$')
axes[1].set_ylabel('Amplitud (parte real)')
axes[1].set_title('Símbolo OFDM con prefijo cíclico')

plt.tight_layout()
plt.savefig('figures/ofdm-subcarriers.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('figures/cp-illustration.png', dpi=300, bbox_inches='tight', facecolor='white')
# Guardar figuras separadas
fig2, ax2 = plt.subplots(figsize=(7, 4))
for idx, sc in enumerate([-2, -1, 0, 1, 2, 3]):
    sinc_env = np.sinc(f - sc)
    ax2.plot(f, np.abs(sinc_env), color=colors[idx], lw=1.5)
ax2.set_xlabel('Frecuencia normalizada ($\\times \\Delta f$)')
ax2.set_ylabel('Amplitud')
ax2.set_title('Subportadoras OFDM: ortogonalidad espectral')
ax2.set_xlim(-3, 3)
plt.tight_layout()
plt.savefig('figures/ofdm-subcarriers.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close('all')

fig3, ax3 = plt.subplots(figsize=(7, 4))
ax3.plot(t_full, x_with_cp, 'b-', lw=1.5)
ax3.axvspan(0, N_CP, alpha=0.25, color='orange', label=f'CP ({N_CP} muestras)')
ax3.axvline(N_CP, color='orange', ls='--', lw=1)
ax3.set_xlabel('Muestra $n$')
ax3.set_ylabel('Amplitud (parte real)')
ax3.set_title('Símbolo OFDM con prefijo cíclico')
ax3.legend()
plt.tight_layout()
plt.savefig('figures/cp-illustration.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close('all')
print('Figuras para apuntes guardadas.')

### Preguntas de reflexión — Ejercicio 5

1. Identifica las subportadoras con mayor BER. ¿Corresponden a las subportadoras con menor ganancia de canal $|H[k]|$?

2. La BER global de OFDM es mayor que la de AWGN para el mismo $E_b/N_0$. ¿Qué técnica de las estudiadas en este curso podría igualarlas?
   > *Pista: si pudieras asignar más potencia a las subportadoras con mala ganancia, o más protección de código a los bits en esas subportadoras...*

3. Implementa un canal con **un solo camino** ($h = [1.0]$) y verifica que la BER de OFDM coincide con la de AWGN puro. Esto confirma que OFDM + ecualización perfecta sobre canal plano es equivalente a AWGN.

---
## Conclusiones

En este laboratorio has construido y verificado un transceptor OFDM completo:

1. **Símbolo OFDM**: la IFFT genera una señal con amplitud variable (gaussiana para N grande) que contiene N subportadoras ortogonales superimpuestas. El CP copia las últimas N_CP muestras al frente del símbolo.

2. **Recuperación perfecta sin canal**: la cadena IFFT → FFT invierte exactamente, con error numérico de orden $10^{-14}$. La ortogonalidad de la DFT garantiza la separación de subportadoras.

3. **CP elimina ISI e ICI**: sin CP, la convolución lineal del canal mezcla símbolos adyacentes y rompe la ortogonalidad. Con CP $\geq L-1$ muestras, la convolución se vuelve circular y el canal aparece como ganancia escalar por subportadora.

4. **ZF vs MMSE**: el ZF es óptimo a SNR alta (donde el ruido es pequeño) pero amplifica el ruido en subportadoras con canal débil. El MMSE lo balancea y da menor BER a SNR baja.

5. **BER por subportadora**: las subportadoras con baja ganancia de canal contribuyen más a la BER global. Técnicas como *link adaptation* por subportadora (water-filling, Sesión 06) y turbo-equalization (Sesión 04) mitigan este efecto.

### Próximos pasos

- **Sesión 04 — Codificación de canal**: el código LDPC opera sobre los bits de todos los RBs juntos — incluyendo los de las subportadoras débiles. La mezcla de bits en el codificador hace que el código "promedio" las subportadoras buenas y malas.
- **Sesión 05 — Acceso múltiple OFDMA**: distintos usuarios reciben subconjuntos de subportadoras. El planificador puede asignar a cada usuario las subportadoras donde tiene mejor canal.
- **Sesión 08 — Estimación de canal**: hasta ahora hemos asumido $H[k]$ perfectamente conocido. En la práctica se estima con pilotos — la imprecisión limita la BER del sistema real.

### Lecturas adicionales

- Proakis & Salehi, *Digital Communications* — Cap. 9: derivación completa de OFDM y CP
- 3GPP TS 38.211 §4: estructura de la ranura NR, numerologías y CP
- Dahlman et al., *5G NR* — Cap. 7: implementación real del transceptor OFDM en 5G NR